In [20]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from itertools import product

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train[train.label == 1].head()


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")

train["title"] = train["title"].fillna("")
train["body"]  = train["body"].fillna("")

train["text"] = train["title"] + " " + train["body"]

dep = train[train["label"] == 1]["text"]
non_dep = train[train["label"] == 0]["text"]

print(train["label"].value_counts())

cv = CountVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=3,
    ngram_range=(1,1)
)

X_dep = cv.fit_transform(dep)

freq = np.asarray(X_dep.sum(axis=0)).ravel()
words = np.array(cv.get_feature_names_out())

top_dep = pd.DataFrame({
    "word": words,
    "count_in_dep": freq
}).sort_values("count_in_dep", ascending=False)

print("\nTOP WORDS IN DEPRESSED CLASS:")
print(top_dep.head(50))

cv2 = CountVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=5,
    ngram_range=(1,1)
)

X_all = cv2.fit_transform(train["text"])
vocab = np.array(cv2.get_feature_names_out())

X1 = cv2.transform(dep)
X0 = cv2.transform(non_dep)

c1 = np.asarray(X1.sum(axis=0)).ravel() + 1
c0 = np.asarray(X0.sum(axis=0)).ravel() + 1

score = np.log(c1 / c1.sum()) - np.log(c0 / c0.sum())

top_signal = pd.DataFrame({
    "word": vocab,
    "log_odds_dep": score,
    "dep_count": c1 - 1,
    "nondep_count": c0 - 1
}).sort_values("log_odds_dep", ascending=False)

print("\nMOST DEPRESSION-SPECIFIC WORDS:")
print(top_signal.head(60))

cv3 = CountVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=3,
    ngram_range=(2,2)
)

X_bi = cv3.fit_transform(dep)

freq_bi = np.asarray(X_bi.sum(axis=0)).ravel()
grams = np.array(cv3.get_feature_names_out())

top_bigrams = pd.DataFrame({
    "bigram": grams,
    "count": freq_bi
}).sort_values("count", ascending=False)

print("\nTOP BIGRAMS IN DEPRESSED CLASS:")
print(top_bigrams.head(50))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


# =========================
# 1. LOAD DATA
# =========================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train["title"] = train["title"].fillna("").astype(str)
train["body"] = train["body"].fillna("").astype(str)

test["title"] = test["title"].fillna("").astype(str)
test["body"] = test["body"].fillna("").astype(str)

y = train["label"].astype(int).values

# =========================
# 2. CV SETUP
# =========================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


# =========================
# 3. HELPER FUNCTION
# =========================
def train_oof_model(
    X_train_full,
    X_test_full,
    y,
    skf,
    feature_name,
    tfidf_params,
    clf_params
):
    oof_pred = np.zeros(len(X_train_full))
    test_pred_folds = np.zeros((len(X_test_full), skf.n_splits))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_full, y), 1):
        X_train = X_train_full.iloc[train_idx]
        X_valid = X_train_full.iloc[valid_idx]
        y_train = y[train_idx]
        y_valid = y[valid_idx]

        model = Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf", LogisticRegression(**clf_params))
        ])

        model.fit(X_train, y_train)

        valid_pred = model.predict_proba(X_valid)[:, 1]
        test_pred = model.predict_proba(X_test_full)[:, 1]

        oof_pred[valid_idx] = valid_pred
        test_pred_folds[:, fold - 1] = test_pred

        fold_f1 = f1_score(y_valid, (valid_pred >= 0.5).astype(int))
        fold_scores.append(fold_f1)

        print(f"[{feature_name}] Fold {fold}: F1 = {fold_f1:.5f}")

    oof_f1_05 = f1_score(y, (oof_pred >= 0.5).astype(int))
    mean_cv_f1 = np.mean(fold_scores)

    best_thr = 0.5
    best_f1 = oof_f1_05

    for thr in np.arange(0.10, 0.91, 0.01):
        pred_bin = (oof_pred >= thr).astype(int)
        score = f1_score(y, pred_bin)
        if score > best_f1:
            best_f1 = score
            best_thr = thr

    test_pred_mean = test_pred_folds.mean(axis=1)

    print(f"\n[{feature_name}] OOF F1 @ 0.50: {oof_f1_05:.5f}")
    print(f"[{feature_name}] Mean CV F1: {mean_cv_f1:.5f}")
    print(f"[{feature_name}] Best threshold: {best_thr:.2f}")
    print(f"[{feature_name}] Best OOF F1: {best_f1:.5f}")
    print("-" * 60)

    return {
        "oof_pred": oof_pred,
        "test_pred": test_pred_mean,
        "fold_scores": fold_scores,
        "oof_f1_05": oof_f1_05,
        "best_thr": best_thr,
        "best_f1": best_f1
    }


# =========================
# 4. MODEL CONFIGS
# =========================

# title model
title_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.95,
    "sublinear_tf": True
}

title_clf_params = {
    "C": 2.0,
    "class_weight": "balanced",
    "max_iter": 5000,
    "solver": "liblinear",
    "random_state": 42
}

# body model
body_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 3),
    "min_df": 5,
    "max_df": 0.95,
    "max_features": 100_000,
    "sublinear_tf": True
}

body_clf_params = {
    "C": 3.0,
    "class_weight": "balanced",
    "max_iter": 10000,
    "solver": "liblinear",
    "random_state": 42
}


# =========================
# 5. TRAIN TITLE MODEL
# =========================
title_res = train_oof_model(
    X_train_full=train["title"],
    X_test_full=test["title"],
    y=y,
    skf=skf,
    feature_name="title",
    tfidf_params=title_tfidf_params,
    clf_params=title_clf_params
)

train["oof_title_lr"] = title_res["oof_pred"]
test["test_pred_title_lr"] = title_res["test_pred"]


# =========================
# 6. TRAIN BODY MODEL
# =========================
body_res = train_oof_model(
    X_train_full=train["body"],
    X_test_full=test["body"],
    y=y,
    skf=skf,
    feature_name="body",
    tfidf_params=body_tfidf_params,
    clf_params=body_clf_params
)

train["oof_body_lr"] = body_res["oof_pred"]
test["test_pred_body_lr"] = body_res["test_pred"]


# =========================
# 7. BLENDING
# =========================
best_blend_score = -1
best_blend_cfg = None
best_blend_oof = None

# вес title, вес body = 1 - weight_title
# body сильнее, так что смотрим маленькие веса title
for weight_title in np.arange(0.00, 0.31, 0.01):
    weight_body = 1.0 - weight_title

    blended_oof = (
        weight_title * train["oof_title_lr"].values +
        weight_body * train["oof_body_lr"].values
    )

    for thr in np.arange(0.30, 0.81, 0.01):
        pred = (blended_oof >= thr).astype(int)
        score = f1_score(y, pred)

        if score > best_blend_score:
            best_blend_score = score
            best_blend_cfg = (round(weight_title, 2), round(weight_body, 2), round(thr, 2))
            best_blend_oof = blended_oof.copy()

print("\nBEST BLEND CONFIG:")
print(f"title_weight = {best_blend_cfg[0]}")
print(f"body_weight  = {best_blend_cfg[1]}")
print(f"threshold    = {best_blend_cfg[2]}")
print(f"blend OOF F1 = {best_blend_score:.5f}")


# =========================
# 8. FINAL TEST BLEND
# =========================
best_w_title, best_w_body, best_thr = best_blend_cfg

blended_test = (
    best_w_title * test["test_pred_title_lr"].values +
    best_w_body * test["test_pred_body_lr"].values
)

test_pred_label = (blended_test >= best_thr).astype(int)


# =========================
# 9. SAVE FILES
# =========================
# OOF preds
train[["id", "label", "oof_title_lr", "oof_body_lr"]].to_csv("oof_title_body_lr.csv", index=False)

# test preds
test[["id", "test_pred_title_lr", "test_pred_body_lr"]].to_csv("test_title_body_lr.csv", index=False)

# blended submission
submission = pd.DataFrame({
    "id": test["id"],
    "label": test_pred_label
})
submission.to_csv("submission_blend_title_body_lr.csv", index=False)

# optional: save blend probs too
test_blend_df = pd.DataFrame({
    "id": test["id"],
    "blend_pred": blended_test
})
test_blend_df.to_csv("test_blend_title_body_lr.csv", index=False)

print("\nSaved:")
print("- oof_title_body_lr.csv")
print("- test_title_body_lr.csv")
print("- submission_blend_title_body_lr.csv")
print("- test_blend_title_body_lr.csv")

Fold 1: F1 = 0.78409
Fold 2: F1 = 0.79527
Fold 3: F1 = 0.78905
Fold 4: F1 = 0.77946
Fold 5: F1 = 0.78121

OOF F1 @ 0.50: 0.78578
Mean CV F1: 0.78582
Best threshold: 0.55
Best OOF F1: 0.78913

Saved:
- submission_title_lr.csv
- oof_title_lr.csv
- test_title_lr.csv


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================
# PARAM SPACE
# =========================
param_grid = {
    "ngram_range": [(1,1), (1,2), (1,3)],
    "min_df": [2, 3, 5, 10],
    "max_df": [0.90, 0.95, 1.0],
    "sublinear_tf": [True, False],
    "max_features": [None, 50000, 80000],
    "C": [0.5, 1.0, 2.0, 3.0, 5.0, 8.0],
    "class_weight": [None, "balanced"],
}

# если хочешь не слишком долго, можно сначала урезать:
# "ngram_range": [(1,2), (1,3)]
# "min_df": [3, 5]
# "max_df": [0.95, 1.0]
# "sublinear_tf": [True]
# "max_features": [None, 50000, 80000]
# "C": [1.0, 2.0, 3.0, 5.0]
# "class_weight": ["balanced"]

all_params = list(product(
    param_grid["ngram_range"],
    param_grid["min_df"],
    param_grid["max_df"],
    param_grid["sublinear_tf"],
    param_grid["max_features"],
    param_grid["C"],
    param_grid["class_weight"],
))

print(f"Total configs: {len(all_params)}")

results = []

def find_best_threshold(y_true, pred_proba, thr_min=0.30, thr_max=0.80, thr_step=0.01):
    best_thr = 0.5
    best_score = -1
    
    thr = thr_min
    while thr <= thr_max + 1e-9:
        pred = (pred_proba >= thr).astype(int)
        score = f1_score(y_true, pred)
        if score > best_score:
            best_score = score
            best_thr = round(thr, 4)
        thr += thr_step
    
    return best_thr, best_score

for i, (ngram_range, min_df, max_df, sublinear_tf, max_features, C, class_weight) in enumerate(all_params, 1):
    oof_pred = np.zeros(len(train))
    fold_scores_05 = []

    try:
        for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
            X_train = X.iloc[train_idx]
            X_valid = X.iloc[valid_idx]
            y_train = y[train_idx]
            y_valid = y[valid_idx]

            model = Pipeline([
                ("tfidf", TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    analyzer="word",
                    ngram_range=ngram_range,
                    min_df=min_df,
                    max_df=max_df,
                    sublinear_tf=sublinear_tf,
                    max_features=max_features
                )),
                ("clf", LogisticRegression(
                    C=C,
                    class_weight=class_weight,
                    max_iter=5000,
                    solver="liblinear",
                    random_state=42
                ))
            ])

            model.fit(X_train, y_train)
            valid_pred = model.predict_proba(X_valid)[:, 1]
            oof_pred[valid_idx] = valid_pred

            fold_f1_05 = f1_score(y_valid, (valid_pred >= 0.5).astype(int))
            fold_scores_05.append(fold_f1_05)

        oof_f1_05 = f1_score(y, (oof_pred >= 0.5).astype(int))
        best_thr, best_oof_f1 = find_best_threshold(y, oof_pred)

        row = {
            "config_id": i,
            "ngram_range": str(ngram_range),
            "min_df": min_df,
            "max_df": max_df,
            "sublinear_tf": sublinear_tf,
            "max_features": max_features,
            "C": C,
            "class_weight": class_weight,
            "mean_fold_f1_05": np.mean(fold_scores_05),
            "oof_f1_05": oof_f1_05,
            "best_thr": best_thr,
            "best_oof_f1": best_oof_f1,
        }
        results.append(row)

        print(
            f"[{i:03d}/{len(all_params)}] "
            f"best_oof_f1={best_oof_f1:.5f} | thr={best_thr:.2f} | "
            f"ngram={ngram_range}, min_df={min_df}, max_df={max_df}, "
            f"sublinear={sublinear_tf}, max_features={max_features}, "
            f"C={C}, class_weight={class_weight}"
        )

    except Exception as e:
        print(f"[{i:03d}/{len(all_params)}] FAILED: {e}")

results_df = pd.DataFrame(results).sort_values("best_oof_f1", ascending=False)
results_df.to_csv("body_lr_tuning_results.csv", index=False)

print("\nTOP 20 CONFIGS:")
print(results_df.head(20))

In [ ]:
# 
# обучим модель на всем train, чтобы посмотреть веса
final_model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=5,#3,
            max_df=0.95,
            max_features=100_000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            C=3.0,#2.0,
            class_weight="balanced",
            max_iter=5000,
            solver="liblinear",
            random_state=42
        ))
    ])

final_model.fit(X, y)

vectorizer = final_model.named_steps["tfidf"]
clf = final_model.named_steps["clf"]

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]

top_pos_idx = np.argsort(coefs)[-30:][::-1]
top_neg_idx = np.argsort(coefs)[:30]

print("\nTOP FEATURES FOR CLASS 1:")
for f, w in zip(feature_names[top_pos_idx], coefs[top_pos_idx]):
    print(f"{f:30s} {w:.4f}")

print("\nTOP FEATURES FOR CLASS 0:")
for f, w in zip(feature_names[top_neg_idx], coefs[top_neg_idx]):
    print(f"{f:30s} {w:.4f}")


TOP FEATURES FOR CLASS 1:
depression                     8.4832
myself                         6.8034
suicide                        5.9086
feel                           5.5761
life                           5.3365
anymore                        4.9558
depressed                      4.7951
die                            4.2325
alone                          4.1059
feeling                        3.8340
suicidal                       3.7691
live                           3.6983
can                            3.6570
wish                           3.6069
anyone                         3.5633
goodbye                        3.5260
still here                     3.5060
job                            3.4725
through                        3.3455
care                           3.2759
nothing                        3.2323
worse                          3.2107
no one                         3.1895
could                          3.1238
years                          3.0736
dead                   

In [12]:
submission = pd.read_csv('submission.csv')
submission.label = submission.label.apply(lambda value: 1 if value > 0.6 else 0)
submission.to_csv('submission_binary.csv', index=False)

In [22]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train["title"] = train["title"].fillna("").astype(str)
train["body"] = train["body"].fillna("").astype(str)

test["title"] = test["title"].fillna("").astype(str)
test["body"] = test["body"].fillna("").astype(str)

y = train["label"].astype(int).values

# =========================
# 2. CV SETUP
# =========================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


# =========================
# 3. HELPER FUNCTION
# =========================
def train_oof_model(
    X_train_full,
    X_test_full,
    y,
    skf,
    feature_name,
    tfidf_params,
    clf_params
):
    oof_pred = np.zeros(len(X_train_full))
    test_pred_folds = np.zeros((len(X_test_full), skf.n_splits))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_full, y), 1):
        X_train = X_train_full.iloc[train_idx]
        X_valid = X_train_full.iloc[valid_idx]
        y_train = y[train_idx]
        y_valid = y[valid_idx]

        model = Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf", LogisticRegression(**clf_params))
        ])

        model.fit(X_train, y_train)

        valid_pred = model.predict_proba(X_valid)[:, 1]
        test_pred = model.predict_proba(X_test_full)[:, 1]

        oof_pred[valid_idx] = valid_pred
        test_pred_folds[:, fold - 1] = test_pred

        fold_f1 = f1_score(y_valid, (valid_pred >= 0.5).astype(int))
        fold_scores.append(fold_f1)

        print(f"[{feature_name}] Fold {fold}: F1 = {fold_f1:.5f}")

    oof_f1_05 = f1_score(y, (oof_pred >= 0.5).astype(int))
    mean_cv_f1 = np.mean(fold_scores)

    best_thr = 0.5
    best_f1 = oof_f1_05

    for thr in np.arange(0.10, 0.91, 0.01):
        pred_bin = (oof_pred >= thr).astype(int)
        score = f1_score(y, pred_bin)
        if score > best_f1:
            best_f1 = score
            best_thr = thr

    test_pred_mean = test_pred_folds.mean(axis=1)

    print(f"\n[{feature_name}] OOF F1 @ 0.50: {oof_f1_05:.5f}")
    print(f"[{feature_name}] Mean CV F1: {mean_cv_f1:.5f}")
    print(f"[{feature_name}] Best threshold: {best_thr:.2f}")
    print(f"[{feature_name}] Best OOF F1: {best_f1:.5f}")
    print("-" * 60)

    return {
        "oof_pred": oof_pred,
        "test_pred": test_pred_mean,
        "fold_scores": fold_scores,
        "oof_f1_05": oof_f1_05,
        "best_thr": best_thr,
        "best_f1": best_f1
    }


# =========================
# 4. MODEL CONFIGS
# =========================

# title model
title_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.95,
    "sublinear_tf": True
}

title_clf_params = {
    "C": 2.0,
    "class_weight": "balanced",
    "max_iter": 5000,
    "solver": "liblinear",
    "random_state": 42
}

# body model
body_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 5,
    "max_df": 0.95,
    "max_features": 100_000,
    "sublinear_tf": True
}

body_clf_params = {
    "C": 3.0,
    "class_weight": "balanced",
    "max_iter": 10000,
    "solver": "liblinear",
    "random_state": 42
}


# =========================
# 5. TRAIN TITLE MODEL
# =========================
title_res = train_oof_model(
    X_train_full=train["title"],
    X_test_full=test["title"],
    y=y,
    skf=skf,
    feature_name="title",
    tfidf_params=title_tfidf_params,
    clf_params=title_clf_params
)

train["oof_title_lr"] = title_res["oof_pred"]
test["test_pred_title_lr"] = title_res["test_pred"]


# =========================
# 6. TRAIN BODY MODEL
# =========================
body_res = train_oof_model(
    X_train_full=train["body"],
    X_test_full=test["body"],
    y=y,
    skf=skf,
    feature_name="body",
    tfidf_params=body_tfidf_params,
    clf_params=body_clf_params
)

train["oof_body_lr"] = body_res["oof_pred"]
test["test_pred_body_lr"] = body_res["test_pred"]


# =========================
# 7. BLENDING
# =========================
best_blend_score = -1
best_blend_cfg = None
best_blend_oof = None

# вес title, вес body = 1 - weight_title
# body сильнее, так что смотрим маленькие веса title
for weight_title in np.arange(0.00, 0.31, 0.01):
    weight_body = 1.0 - weight_title

    blended_oof = (
        weight_title * train["oof_title_lr"].values +
        weight_body * train["oof_body_lr"].values
    )

    for thr in np.arange(0.30, 0.81, 0.01):
        pred = (blended_oof >= thr).astype(int)
        score = f1_score(y, pred)

        if score > best_blend_score:
            best_blend_score = score
            best_blend_cfg = (round(weight_title, 2), round(weight_body, 2), round(thr, 2))
            best_blend_oof = blended_oof.copy()

print("\nBEST BLEND CONFIG:")
print(f"title_weight = {best_blend_cfg[0]}")
print(f"body_weight  = {best_blend_cfg[1]}")
print(f"threshold    = {best_blend_cfg[2]}")
print(f"blend OOF F1 = {best_blend_score:.5f}")


# =========================
# 8. FINAL TEST BLEND
# =========================
best_w_title, best_w_body, best_thr = best_blend_cfg

blended_test = (
    best_w_title * test["test_pred_title_lr"].values +
    best_w_body * test["test_pred_body_lr"].values
)

test_pred_label = (blended_test >= best_thr).astype(int)


# =========================
# 9. SAVE FILES
# =========================
# OOF preds
train[["id", "label", "oof_title_lr", "oof_body_lr"]].to_csv("oof_title_body_lr.csv", index=False)

# test preds
test[["id", "test_pred_title_lr", "test_pred_body_lr"]].to_csv("test_title_body_lr.csv", index=False)

# blended submission
submission = pd.DataFrame({
    "id": test["id"],
    "label": test_pred_label
})
submission.to_csv("submission_blend_title_body_lr.csv", index=False)

# optional: save blend probs too
test_blend_df = pd.DataFrame({
    "id": test["id"],
    "blend_pred": blended_test
})
test_blend_df.to_csv("test_blend_title_body_lr.csv", index=False)

print("\nSaved:")
print("- oof_title_body_lr.csv")
print("- test_title_body_lr.csv")
print("- submission_blend_title_body_lr.csv")
print("- test_blend_title_body_lr.csv")

[title] Fold 1: F1 = 0.61246
[title] Fold 2: F1 = 0.58785
[title] Fold 3: F1 = 0.59676
[title] Fold 4: F1 = 0.60203
[title] Fold 5: F1 = 0.60988

[title] OOF F1 @ 0.50: 0.60190
[title] Mean CV F1: 0.60180
[title] Best threshold: 0.53
[title] Best OOF F1: 0.60653
------------------------------------------------------------
[body] Fold 1: F1 = 0.78121
[body] Fold 2: F1 = 0.79004
[body] Fold 3: F1 = 0.78857
[body] Fold 4: F1 = 0.78413
[body] Fold 5: F1 = 0.78341

[body] OOF F1 @ 0.50: 0.78545
[body] Mean CV F1: 0.78547
[body] Best threshold: 0.59
[body] Best OOF F1: 0.79020
------------------------------------------------------------

BEST BLEND CONFIG:
title_weight = 0.3
body_weight  = 0.7
threshold    = 0.51
blend OOF F1 = 0.80600

Saved:
- oof_title_body_lr.csv
- test_title_body_lr.csv
- submission_blend_title_body_lr.csv
- test_blend_title_body_lr.csv


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# =========================
# 1. LOAD DATA
# =========================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train["title"] = train["title"].fillna("").astype(str)
train["body"] = train["body"].fillna("").astype(str)

test["title"] = test["title"].fillna("").astype(str)
test["body"] = test["body"].fillna("").astype(str)

train["full_text"] = train["title"] + " " + train["body"]
test["full_text"] = test["title"] + " " + test["body"]

X = train["full_text"]
y = train["label"].astype(int).values
X_test = test["full_text"]

# =========================
# 2. CV
# =========================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred = np.zeros(len(train))
test_pred_folds = np.zeros((len(test), n_splits))
fold_scores = []

# =========================
# 3. MODEL
# =========================
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="char_wb",
            ngram_range=(3, 6),
            min_df=3,
            max_df=0.95,
            max_features=250_000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            C=3.0,
            class_weight="balanced",
            max_iter=15000,
            solver="liblinear",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    oof_pred[valid_idx] = valid_pred
    test_pred_folds[:, fold - 1] = test_pred

    valid_f1 = f1_score(y_valid, (valid_pred >= 0.5).astype(int))
    fold_scores.append(valid_f1)

    print(f"Fold {fold}: F1 = {valid_f1:.5f}")

# =========================
# 4. OOF SCORE
# =========================
oof_f1_05 = f1_score(y, (oof_pred >= 0.5).astype(int))
print("\nOOF F1 @ 0.50:", round(oof_f1_05, 5))
print("Mean CV F1:", round(np.mean(fold_scores), 5))

# =========================
# 5. THRESHOLD SEARCH
# =========================
best_thr = 0.5
best_f1 = oof_f1_05

for thr in np.arange(0.10, 0.91, 0.01):
    pred_bin = (oof_pred >= thr).astype(int)
    score = f1_score(y, pred_bin)
    if score > best_f1:
        best_f1 = score
        best_thr = thr

print(f"Best threshold: {best_thr:.2f}")
print(f"Best OOF F1: {best_f1:.5f}")

# =========================
# 6. SAVE
# =========================
test_pred_mean = test_pred_folds.mean(axis=1)

train["oof_char_lr"] = oof_pred
test["test_pred_char_lr"] = test_pred_mean

train[["id", "label", "oof_char_lr"]].to_csv("oof_char_lr.csv", index=False)
test[["id", "test_pred_char_lr"]].to_csv("test_char_lr.csv", index=False)

submission = pd.DataFrame({
    "id": test["id"],
    "label": (test_pred_mean >= best_thr).astype(int)
})
submission.to_csv("submission_char_lr.csv", index=False)

print("\nSaved:")
print("- oof_char_lr.csv")
print("- test_char_lr.csv")
print("- submission_char_lr.csv")

Fold 1: F1 = 0.81591
Fold 2: F1 = 0.82039
Fold 3: F1 = 0.82207
Fold 4: F1 = 0.83105
Fold 5: F1 = 0.81933

OOF F1 @ 0.50: 0.82176
Mean CV F1: 0.82175
Best threshold: 0.53
Best OOF F1: 0.82317

Saved:
- oof_char_lr.csv
- test_char_lr.csv
- submission_char_lr.csv


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train["title"] = train["title"].fillna("").astype(str)
train["body"] = train["body"].fillna("").astype(str)

test["title"] = test["title"].fillna("").astype(str)
test["body"] = test["body"].fillna("").astype(str)

train["full_text"] = train["title"] + " " + train["body"]
test["full_text"] = test["title"] + " " + test["body"]

y = train["label"].astype(int).values

# =========================
# CV SETUP
# =========================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


# =========================
# HELPER FUNCTION
# =========================
def train_oof_model(
    X_train_full,
    X_test_full,
    y,
    skf,
    feature_name,
    tfidf_params,
    clf_params
):
    oof_pred = np.zeros(len(X_train_full))
    test_pred_folds = np.zeros((len(X_test_full), skf.n_splits))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_full, y), 1):
        X_train = X_train_full.iloc[train_idx]
        X_valid = X_train_full.iloc[valid_idx]
        y_train = y[train_idx]
        y_valid = y[valid_idx]

        model = Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf", LogisticRegression(**clf_params))
        ])

        model.fit(X_train, y_train)

        valid_pred = model.predict_proba(X_valid)[:, 1]
        test_pred = model.predict_proba(X_test_full)[:, 1]

        oof_pred[valid_idx] = valid_pred
        test_pred_folds[:, fold - 1] = test_pred

        fold_f1 = f1_score(y_valid, (valid_pred >= 0.5).astype(int))
        fold_scores.append(fold_f1)

        print(f"[{feature_name}] Fold {fold}: F1 = {fold_f1:.5f}")

    oof_f1_05 = f1_score(y, (oof_pred >= 0.5).astype(int))
    mean_cv_f1 = np.mean(fold_scores)

    best_thr = 0.5
    best_f1 = oof_f1_05

    for thr in np.arange(0.10, 0.91, 0.01):
        pred_bin = (oof_pred >= thr).astype(int)
        score = f1_score(y, pred_bin)
        if score > best_f1:
            best_f1 = score
            best_thr = thr

    test_pred_mean = test_pred_folds.mean(axis=1)

    print(f"\n[{feature_name}] OOF F1 @ 0.50: {oof_f1_05:.5f}")
    print(f"[{feature_name}] Mean CV F1: {mean_cv_f1:.5f}")
    print(f"[{feature_name}] Best threshold: {best_thr:.2f}")
    print(f"[{feature_name}] Best OOF F1: {best_f1:.5f}")
    print("-" * 60)

    return {
        "oof_pred": oof_pred,
        "test_pred": test_pred_mean,
        "fold_scores": fold_scores,
        "oof_f1_05": oof_f1_05,
        "best_thr": best_thr,
        "best_f1": best_f1
    }


# =========================
# MODEL CONFIGS
# =========================

# title model
title_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.95,
    "sublinear_tf": True
}

title_clf_params = {
    "C": 2.0,
    "class_weight": "balanced",
    "max_iter": 5000,
    "solver": "liblinear",
    "random_state": 42
}

# body model
body_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 5,
    "max_df": 0.95,
    "max_features": 100_000,
    "sublinear_tf": True
}

body_clf_params = {
    "C": 3.0,
    "class_weight": "balanced",
    "max_iter": 10000,
    "solver": "liblinear",
    "random_state": 42
}

# char model
char_tfidf_params = {
    "lowercase": True,
    "strip_accents": "unicode",
    "analyzer": "char_wb",
    "ngram_range": (3, 5),
    "min_df": 3,
    "max_df": 0.95,
    "max_features": 200_000,
    "sublinear_tf": True
}

char_clf_params = {
    "C": 3.0,
    "class_weight": "balanced",
    "max_iter": 10000,
    "solver": "liblinear",
    "random_state": 42
}


# =========================
# TRAIN TITLE MODEL
# =========================
title_res = train_oof_model(
    X_train_full=train["title"],
    X_test_full=test["title"],
    y=y,
    skf=skf,
    feature_name="title",
    tfidf_params=title_tfidf_params,
    clf_params=title_clf_params
)

train["oof_title_lr"] = title_res["oof_pred"]
test["test_pred_title_lr"] = title_res["test_pred"]


# =========================
# TRAIN BODY MODEL
# =========================
body_res = train_oof_model(
    X_train_full=train["body"],
    X_test_full=test["body"],
    y=y,
    skf=skf,
    feature_name="body",
    tfidf_params=body_tfidf_params,
    clf_params=body_clf_params
)

train["oof_body_lr"] = body_res["oof_pred"]
test["test_pred_body_lr"] = body_res["test_pred"]


# =========================
# TRAIN CHAR MODEL
# =========================
char_res = train_oof_model(
    X_train_full=train["full_text"],
    X_test_full=test["full_text"],
    y=y,
    skf=skf,
    feature_name="char_full_text",
    tfidf_params=char_tfidf_params,
    clf_params=char_clf_params
)

train["oof_char_lr"] = char_res["oof_pred"]
test["test_pred_char_lr"] = char_res["test_pred"]


# # =========================
# # 3-MODEL BLENDING
# # =========================
# best_blend_score = -1
# best_blend_cfg = None
# best_blend_oof = None

# # ожидаем, что char самый сильный,
# # body второй,
# # title третий
# for w_title in np.arange(0.00, 0.21, 0.01):
#     for w_body in np.arange(0.00, 0.51, 0.01):
#         w_char = 1.0 - w_title - w_body

#         if w_char < 0:
#             continue

#         # чтобы не получилась бессмысленная смесь,
#         # где лучшей модели дали слишком маленький вес
#         if w_char < 0.40:
#             continue

#         blended_oof = (
#             w_title * train["oof_title_lr"].values +
#             w_body * train["oof_body_lr"].values +
#             w_char * train["oof_char_lr"].values
#         )

#         for thr in np.arange(0.30, 0.81, 0.01):
#             pred = (blended_oof >= thr).astype(int)
#             score = f1_score(y, pred)

#             if score > best_blend_score:
#                 best_blend_score = score
#                 best_blend_cfg = (
#                     round(w_title, 2),
#                     round(w_body, 2),
#                     round(w_char, 2),
#                     round(thr, 2)
#                 )
#                 best_blend_oof = blended_oof.copy()

# print("\nBEST 3-MODEL BLEND CONFIG:")
# print(f"title_weight = {best_blend_cfg[0]}")
# print(f"body_weight  = {best_blend_cfg[1]}")
# print(f"char_weight  = {best_blend_cfg[2]}")
# print(f"threshold    = {best_blend_cfg[3]}")
# print(f"blend OOF F1 = {best_blend_score:.5f}")


# =========================
# FINAL TEST BLEND
# =========================
best_w_title, best_w_body, best_w_char, best_thr = best_blend_cfg

blended_test = (
    best_w_title * test["test_pred_title_lr"].values +
    best_w_body * test["test_pred_body_lr"].values +
    best_w_char * test["test_pred_char_lr"].values
)

test_pred_label = (blended_test >= best_thr).astype(int)


# =========================
# SAVE FILES
# =========================
# train[
#     ["id", "label", "oof_title_lr", "oof_body_lr", "oof_char_lr"]
# ].to_csv("oof_title_body_char_lr.csv", index=False)

# test[
#     ["id", "test_pred_title_lr", "test_pred_body_lr", "test_pred_char_lr"]
# ].to_csv("test_title_body_char_lr.csv", index=False)

# submission = pd.DataFrame({
#     "id": test["id"],
#     "label": test_pred_label
# })
# submission.to_csv("submission_blend_title_body_char_lr.csv", index=False)

# test_blend_df = pd.DataFrame({
#     "id": test["id"],
#     "blend_pred": blended_test
# })
# test_blend_df.to_csv("test_blend_title_body_char_lr.csv", index=False)

# print("\nSaved:")
# print("- oof_title_body_char_lr.csv")
# print("- test_title_body_char_lr.csv")
# print("- submission_blend_title_body_char_lr.csv")
# print("- test_blend_title_body_char_lr.csv")

[title] Fold 1: F1 = 0.61246
[title] Fold 2: F1 = 0.58785
[title] Fold 3: F1 = 0.59676
[title] Fold 4: F1 = 0.60203
[title] Fold 5: F1 = 0.60988

[title] OOF F1 @ 0.50: 0.60190
[title] Mean CV F1: 0.60180
[title] Best threshold: 0.53
[title] Best OOF F1: 0.60653
------------------------------------------------------------
[body] Fold 1: F1 = 0.78121
[body] Fold 2: F1 = 0.79004
[body] Fold 3: F1 = 0.78857
[body] Fold 4: F1 = 0.78413
[body] Fold 5: F1 = 0.78341

[body] OOF F1 @ 0.50: 0.78545
[body] Mean CV F1: 0.78547
[body] Best threshold: 0.59
[body] Best OOF F1: 0.79020
------------------------------------------------------------
[char_full_text] Fold 1: F1 = 0.81591
[char_full_text] Fold 2: F1 = 0.82039
[char_full_text] Fold 3: F1 = 0.82207
[char_full_text] Fold 4: F1 = 0.83105
[char_full_text] Fold 5: F1 = 0.81933

[char_full_text] OOF F1 @ 0.50: 0.82176
[char_full_text] Mean CV F1: 0.82175
[char_full_text] Best threshold: 0.53
[char_full_text] Best OOF F1: 0.82317
-----------------

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler


# =========================
# 1. LOAD
# =========================
# train = pd.read_csv("train.csv")
# test = pd.read_csv("test.csv")

# если у тебя уже сохранены OOF / test preds, можно читать их отдельно:
# oof_preds = pd.read_csv("oof_title_body_char_lr.csv")
# test_preds = pd.read_csv("test_title_body_char_lr.csv")

# а можно, если ты работаешь в одном ноутбуке/скрипте, использовать те же train/test,
# в которые ты уже записал oof_title_lr, oof_body_lr, oof_char_lr и test_pred_*


# =========================
# 2. PREPROCESS
# =========================
for col in ["title", "body"]:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)

train["full_text"] = train["title"] + " " + train["body"]
test["full_text"] = test["title"] + " " + test["body"]

y = train["label"].astype(int).values


# =========================
# 3. LEXICONS
# =========================
SUICIDE_PHRASES = [
    "kill myself", "want die", "want to die", "want kill", "end it", "end my life"
]

SUICIDE_WORDS = {
    "suicide", "suicidal", "overdose", "overdosing", "die", "dead", "death"
}

DEPRESSION_WORDS = {
    "depression", "depressed", "therapist", "therapy", "meds", "medication",
    "antidepressant", "psychiatrist", "psychologist", "anxiety", "ptsd", "lexapro"
}

HOPELESS_WORDS = {
    "nothing", "numb", "empty", "emptiness", "hopeless", "miserable", "exhausted",
    "drained", "pointless", "worthless", "alone", "lonely", "tired", "guilt"
}

DISTRESS_WORDS = {
    "hate", "pain", "cry", "crying", "broken", "fucked", "useless", "failure",
    "cant", "cannot", "don't", "dont", "can't"
}

FIRST_PERSON_WORDS = {"i", "me", "my", "myself", "im", "i'm"}
NEGATION_WORDS = {"no", "not", "never", "nothing", "none", "nobody", "nowhere", "dont", "don't", "cant", "can't", "cannot"}


# =========================
# 4. HELPERS
# =========================
_word_re = re.compile(r"\b\w+\b", flags=re.UNICODE)

def safe_div(a, b):
    return a / b if b != 0 else 0.0

def count_words(text):
    return _word_re.findall(text.lower())

def count_substring_occurrences(text, phrases):
    text_low = text.lower()
    return sum(text_low.count(p) for p in phrases)

def count_token_hits(tokens, vocab):
    return sum(1 for t in tokens if t in vocab)

def count_char(text, ch):
    return text.count(ch)

def count_ellipsis(text):
    return text.count("...")

def ratio_upper(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    uppers = sum(1 for c in letters if c.isupper())
    return uppers / len(letters)

def ratio_digits(text):
    if len(text) == 0:
        return 0.0
    return sum(c.isdigit() for c in text) / len(text)

def build_features(df):
    rows = []

    for _, row in df.iterrows():
        title = row["title"]
        body = row["body"]
        full = row["full_text"]

        title_tokens = count_words(title)
        body_tokens = count_words(body)
        full_tokens = count_words(full)

        title_nw = len(title_tokens)
        body_nw = len(body_tokens)
        full_nw = len(full_tokens)

        suicide_phrase_cnt = count_substring_occurrences(full, SUICIDE_PHRASES)
        suicide_word_cnt = count_token_hits(full_tokens, SUICIDE_WORDS)
        depression_cnt = count_token_hits(full_tokens, DEPRESSION_WORDS)
        hopeless_cnt = count_token_hits(full_tokens, HOPELESS_WORDS)
        distress_cnt = count_token_hits(full_tokens, DISTRESS_WORDS)
        first_person_cnt = count_token_hits(full_tokens, FIRST_PERSON_WORDS)
        negation_cnt = count_token_hits(full_tokens, NEGATION_WORDS)

        rows.append({
            # lengths
            "title_len_chars": len(title),
            "body_len_chars": len(body),
            "full_len_chars": len(full),

            "title_len_words": title_nw,
            "body_len_words": body_nw,
            "full_len_words": full_nw,

            "has_body": int(len(body.strip()) > 0),

            # punctuation
            "title_qmarks": count_char(title, "?"),
            "body_qmarks": count_char(body, "?"),
            "full_qmarks": count_char(full, "?"),

            "title_exclaims": count_char(title, "!"),
            "body_exclaims": count_char(body, "!"),
            "full_exclaims": count_char(full, "!"),

            "title_ellipses": count_ellipsis(title),
            "body_ellipses": count_ellipsis(body),
            "full_ellipses": count_ellipsis(full),

            "newline_count": full.count("\n"),

            # style
            "upper_ratio_full": ratio_upper(full),
            "digit_ratio_full": ratio_digits(full),

            # lexicon counts
            "suicide_phrase_cnt": suicide_phrase_cnt,
            "suicide_word_cnt": suicide_word_cnt,
            "depression_cnt": depression_cnt,
            "hopeless_cnt": hopeless_cnt,
            "distress_cnt": distress_cnt,
            "first_person_cnt": first_person_cnt,
            "negation_cnt": negation_cnt,

            # ratios
            "suicide_phrase_ratio": safe_div(suicide_phrase_cnt, full_nw),
            "suicide_word_ratio": safe_div(suicide_word_cnt, full_nw),
            "depression_ratio": safe_div(depression_cnt, full_nw),
            "hopeless_ratio": safe_div(hopeless_cnt, full_nw),
            "distress_ratio": safe_div(distress_cnt, full_nw),
            "first_person_ratio": safe_div(first_person_cnt, full_nw),
            "negation_ratio": safe_div(negation_cnt, full_nw),

            # title/body relations
            "title_to_body_len_ratio": safe_div(len(title), len(body) + 1),
            "title_to_body_word_ratio": safe_div(title_nw, body_nw + 1),
        })

    return pd.DataFrame(rows)


# =========================
# 5. BUILD DENSE FEATURES
# =========================
X_dense_train = build_features(train)
X_dense_test = build_features(test)

print("Dense train shape:", X_dense_train.shape)
print("Dense test shape:", X_dense_test.shape)


# =========================
# 6. OOF FOR DENSE MODEL
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_dense = np.zeros(len(train))
test_dense_folds = np.zeros((len(test), skf.n_splits))
dense_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dense_train, y), 1):
    X_tr = X_dense_train.iloc[tr_idx].copy()
    X_va = X_dense_train.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_test_scaled = scaler.transform(X_dense_test)

    clf = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    )

    clf.fit(X_tr_scaled, y_tr)

    va_pred = clf.predict_proba(X_va_scaled)[:, 1]
    te_pred = clf.predict_proba(X_test_scaled)[:, 1]

    oof_dense[va_idx] = va_pred
    test_dense_folds[:, fold - 1] = te_pred

    fold_f1 = f1_score(y_va, (va_pred >= 0.5).astype(int))
    dense_fold_scores.append(fold_f1)

    print(f"[dense] Fold {fold}: F1 = {fold_f1:.5f}")

dense_test_pred = test_dense_folds.mean(axis=1)

dense_best_thr = 0.5
dense_best_f1 = f1_score(y, (oof_dense >= 0.5).astype(int))

for thr in np.arange(0.10, 0.91, 0.01):
    score = f1_score(y, (oof_dense >= thr).astype(int))
    if score > dense_best_f1:
        dense_best_f1 = score
        dense_best_thr = thr

print("\n[dense] OOF F1 @ 0.50:", round(f1_score(y, (oof_dense >= 0.5).astype(int)), 5))
print("[dense] Mean CV F1:", round(np.mean(dense_fold_scores), 5))
print("[dense] Best threshold:", round(dense_best_thr, 2))
print("[dense] Best OOF F1:", round(dense_best_f1, 5))

train["oof_dense_lr"] = oof_dense
test["test_pred_dense_lr"] = dense_test_pred


# =========================
# 7. META FEATURES
# =========================
# ВАЖНО:
# здесь предполагается, что у тебя уже есть эти колонки в train/test:
# oof_title_lr, oof_body_lr, oof_char_lr
# test_pred_title_lr, test_pred_body_lr, test_pred_char_lr

meta_train = pd.DataFrame({
    "title_pred": train["oof_title_lr"],
    "body_pred": train["oof_body_lr"],
    "char_pred": train["oof_char_lr"],
    "dense_pred": train["oof_dense_lr"],

    # можно докинуть часть самих dense features прямо в мету
    "has_body": X_dense_train["has_body"],
    "body_len_words": X_dense_train["body_len_words"],
    "full_qmarks": X_dense_train["full_qmarks"],
    "full_exclaims": X_dense_train["full_exclaims"],
    "suicide_phrase_cnt": X_dense_train["suicide_phrase_cnt"],
    "suicide_word_cnt": X_dense_train["suicide_word_cnt"],
    "depression_cnt": X_dense_train["depression_cnt"],
    "hopeless_cnt": X_dense_train["hopeless_cnt"],
    "first_person_ratio": X_dense_train["first_person_ratio"],
    "negation_ratio": X_dense_train["negation_ratio"],
})

meta_test = pd.DataFrame({
    "title_pred": test["test_pred_title_lr"],
    "body_pred": test["test_pred_body_lr"],
    "char_pred": test["test_pred_char_lr"],
    "dense_pred": test["test_pred_dense_lr"],

    "has_body": X_dense_test["has_body"],
    "body_len_words": X_dense_test["body_len_words"],
    "full_qmarks": X_dense_test["full_qmarks"],
    "full_exclaims": X_dense_test["full_exclaims"],
    "suicide_phrase_cnt": X_dense_test["suicide_phrase_cnt"],
    "suicide_word_cnt": X_dense_test["suicide_word_cnt"],
    "depression_cnt": X_dense_test["depression_cnt"],
    "hopeless_cnt": X_dense_test["hopeless_cnt"],
    "first_person_ratio": X_dense_test["first_person_ratio"],
    "negation_ratio": X_dense_test["negation_ratio"],
})


# =========================
# 8. META MODEL OOF
# =========================
oof_meta = np.zeros(len(train))
test_meta_folds = np.zeros((len(test), skf.n_splits))
meta_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(meta_train, y), 1):
    X_tr = meta_train.iloc[tr_idx].copy()
    X_va = meta_train.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_test_scaled = scaler.transform(meta_test)

    meta_clf = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    )

    meta_clf.fit(X_tr_scaled, y_tr)

    va_pred = meta_clf.predict_proba(X_va_scaled)[:, 1]
    te_pred = meta_clf.predict_proba(X_test_scaled)[:, 1]

    oof_meta[va_idx] = va_pred
    test_meta_folds[:, fold - 1] = te_pred

    fold_f1 = f1_score(y_va, (va_pred >= 0.5).astype(int))
    meta_fold_scores.append(fold_f1)

    print(f"[meta] Fold {fold}: F1 = {fold_f1:.5f}")

meta_test_pred = test_meta_folds.mean(axis=1)

meta_best_thr = 0.5
meta_best_f1 = f1_score(y, (oof_meta >= 0.5).astype(int))

for thr in np.arange(0.10, 0.91, 0.01):
    score = f1_score(y, (oof_meta >= thr).astype(int))
    if score > meta_best_f1:
        meta_best_f1 = score
        meta_best_thr = thr

print("\n[meta] OOF F1 @ 0.50:", round(f1_score(y, (oof_meta >= 0.5).astype(int)), 5))
print("[meta] Mean CV F1:", round(np.mean(meta_fold_scores), 5))
print("[meta] Best threshold:", round(meta_best_thr, 2))
print("[meta] Best OOF F1:", round(meta_best_f1, 5))


# =========================
# 9. SAVE
# =========================
train["oof_dense_lr"] = oof_dense
train["oof_meta_lr"] = oof_meta

test["test_pred_dense_lr"] = dense_test_pred
test["test_pred_meta_lr"] = meta_test_pred

submission = pd.DataFrame({
    "id": test["id"],
    "label": (meta_test_pred >= meta_best_thr).astype(int)
})
submission.to_csv("submission_meta_lr.csv", index=False)

train[["id", "label", "oof_dense_lr", "oof_meta_lr"]].to_csv("oof_dense_meta_lr.csv", index=False)
test[["id", "test_pred_dense_lr", "test_pred_meta_lr"]].to_csv("test_dense_meta_lr.csv", index=False)

print("\nSaved:")
print("- submission_meta_lr.csv")
print("- oof_dense_meta_lr.csv")
print("- test_dense_meta_lr.csv")

Dense train shape: (10800, 35)
Dense test shape: (973, 35)
[dense] Fold 1: F1 = 0.71259
[dense] Fold 2: F1 = 0.75113
[dense] Fold 3: F1 = 0.75223
[dense] Fold 4: F1 = 0.70549
[dense] Fold 5: F1 = 0.73767

[dense] OOF F1 @ 0.50: 0.73155
[dense] Mean CV F1: 0.73182
[dense] Best threshold: 0.59
[dense] Best OOF F1: 0.74012


KeyError: 'oof_title_lr'